# 🏗️ ADOIT Data Product — Setup & Infrastructure## Domain: Enterprise Architecture | Catalog: `adoit_product` | Owner: EA TeamThis notebook creates the complete infrastructure for the **ADOIT Data Product** — a self-contained, independently deployable data product following Data Mesh principles.**Self-Contained Data Product Characteristics:**| Characteristic | Implementation ||---|---|| ✅ **Discoverable** | Tags on catalog, schemas, tables; rich COMMENT metadata || ✅ **Addressable** | `adoit_product.gold.application_landscape` — fully qualified || ✅ **Trustworthy** | Quality checks, data contracts, SLA monitoring || ✅ **Self-Describing** | Every column has a COMMENT; TBLPROPERTIES with owner, domain || ✅ **Interoperable** | Delta format, standard schemas, Delta Sharing || ✅ **Secure** | CDF for audit, UDFs for column masking & row filtering |> **This domain operates independently** — no dependency on the Data Mesh Hub.

## Step 1: Create Catalog & SchemasThe EA team owns the entire `adoit_product` catalog. Schemas follow **Medallion Architecture** plus a **Governance** schema.

In [ ]:
%sqlCREATE CATALOG IF NOT EXISTS adoit_product MANAGED LOCATION 'abfss://data-products@sadpsddbxdev.dfs.core.windows.net/adoit_product'COMMENT 'ADOIT Data Product Catalog | Owner: Enterprise Architecture Team | Domain: EA | Source: ADOIT';CREATE SCHEMA IF NOT EXISTS adoit_product.bronze  COMMENT 'Raw Zone | Ingested from ADOIT EA tool | Owner: EA Team';CREATE SCHEMA IF NOT EXISTS adoit_product.silver  COMMENT 'Curated Zone | Cleansed and enriched | Owner: EA Team';CREATE SCHEMA IF NOT EXISTS adoit_product.gold    COMMENT 'Product Zone | Business-ready data products | Owner: EA Team';CREATE SCHEMA IF NOT EXISTS adoit_product.governance COMMENT 'Governance Zone | Quality, contracts, observability | Owner: EA Team';

## Step 2: Create Bronze TablesRaw ingestion targets for ADOIT REST API data.

In [ ]:
%sqlCREATE OR REPLACE TABLE adoit_product.bronze.raw_applications (  app_id STRING COMMENT 'Application identifier (APP-XXX)',  app_name STRING COMMENT 'Application display name',  app_owner STRING COMMENT 'Application owner / responsible person',  department STRING COMMENT 'Owning department',  lifecycle_status STRING COMMENT 'Lifecycle: Active, Retiring, Planned',  criticality STRING COMMENT 'Business criticality: Critical, High, Medium, Low',  technology_stack STRING COMMENT 'Technology platform description',  go_live_date DATE COMMENT 'Initial production deployment date') COMMENT 'Raw application register from ADOIT EA tool'TBLPROPERTIES ('domain' = 'EA', 'owner' = 'EA Team', 'layer' = 'bronze', 'source_system' = 'ADOIT');CREATE OR REPLACE TABLE adoit_product.bronze.raw_business_capabilities (  capability_id STRING COMMENT 'Capability identifier (CAP-XXX)',  capability_name STRING COMMENT 'Business capability name',  capability_domain STRING COMMENT 'Business domain this capability belongs to',  business_value STRING COMMENT 'Value rating: Critical, High, Medium, Low',  maturity_level STRING COMMENT 'Maturity: Advanced, Established, Emerging') COMMENT 'Business capability taxonomy from ADOIT'TBLPROPERTIES ('domain' = 'EA', 'owner' = 'EA Team', 'layer' = 'bronze', 'source_system' = 'ADOIT');CREATE OR REPLACE TABLE adoit_product.bronze.raw_app_capability_map (  app_id STRING COMMENT 'FK to raw_applications',  capability_id STRING COMMENT 'FK to raw_business_capabilities',  support_level STRING COMMENT 'How the app supports the capability: Primary, Secondary') COMMENT 'Mapping between applications and business capabilities'TBLPROPERTIES ('domain' = 'EA', 'owner' = 'EA Team', 'layer' = 'bronze', 'source_system' = 'ADOIT');

## Step 3: Create Silver Tables

In [ ]:
%sqlCREATE OR REPLACE TABLE adoit_product.silver.applications (  app_id STRING, app_name STRING, app_owner STRING, department STRING,  lifecycle_status STRING, criticality STRING, technology_stack STRING,  go_live_date DATE, app_age_years DOUBLE, is_cloud_hosted BOOLEAN,  quality_score DOUBLE, processed_at TIMESTAMP) COMMENT 'Curated applications — enriched with age, cloud flag, quality score'TBLPROPERTIES ('domain' = 'EA', 'owner' = 'EA Team', 'layer' = 'silver');CREATE OR REPLACE TABLE adoit_product.silver.business_capabilities (  capability_id STRING, capability_name STRING, capability_domain STRING,  business_value STRING, maturity_level STRING, ingested_at TIMESTAMP) COMMENT 'Curated business capability taxonomy'TBLPROPERTIES ('domain' = 'EA', 'owner' = 'EA Team', 'layer' = 'silver');

## Step 4: Create Gold Table — Application Landscape Data ProductThe **Application Landscape** joins applications with business capabilities for a 360-degree portfolio view.

In [ ]:
%sqlCREATE OR REPLACE TABLE adoit_product.gold.application_landscape (  app_id STRING COMMENT 'Application identifier',  app_name STRING COMMENT 'Application name',  app_owner STRING COMMENT 'Application owner',  department STRING COMMENT 'Owning department',  lifecycle_status STRING COMMENT 'Lifecycle: Active, Retiring, Planned',  criticality STRING COMMENT 'Business criticality level',  technology_stack STRING COMMENT 'Technology platform',  is_cloud_hosted BOOLEAN COMMENT 'Whether hosted in cloud',  app_age_years DOUBLE COMMENT 'Years since go-live',  capability_count INT COMMENT 'Number of business capabilities supported',  primary_capabilities STRING COMMENT 'Primary capabilities (comma-separated)',  all_capabilities STRING COMMENT 'All capabilities (comma-separated)',  quality_score DOUBLE COMMENT 'Data quality score (0-1.0)',  product_generated_at TIMESTAMP COMMENT 'When this product was last refreshed') COMMENT 'Application Landscape — 360-degree portfolio view with capabilities | SLA: 4h freshness | Quality: >95%'TBLPROPERTIES (  'domain' = 'EA', 'owner' = 'EA Team', 'layer' = 'gold',  'data_product' = 'true', 'sla_freshness_hours' = '4',  'quality_threshold_pct' = '95', 'delta.enableChangeDataFeed' = 'true');

## Step 5: Create Governance Tables (Self-Contained)

In [ ]:
%sqlCREATE OR REPLACE TABLE adoit_product.governance.data_product_registry (  product_id STRING, product_name STRING, domain STRING, owner_group STRING,  owner_contact STRING, table_name STRING, status STRING,  sla_freshness_hours INT, quality_threshold_pct DOUBLE,  quality_score DOUBLE, freshness_status STRING, row_count LONG,  schema_version STRING, consumers INT, created_at TIMESTAMP, updated_at TIMESTAMP,  share_name STRING, listing_status STRING) COMMENT 'Registry of ADOIT domain data products';CREATE OR REPLACE TABLE adoit_product.governance.data_contracts (  contract_id STRING, product_name STRING, producer_group STRING,  consumer_group STRING, contract_status STRING, agreed_sla_hours INT,  quality_threshold_pct DOUBLE, schema_version STRING,  retention_days INT, escalation_contact STRING,  created_at TIMESTAMP, updated_at TIMESTAMP) COMMENT 'Data contracts between ADOIT domain and consumers';CREATE OR REPLACE TABLE adoit_product.governance.data_quality_log (  check_id STRING, check_timestamp TIMESTAMP, data_product STRING,  domain STRING, table_name STRING, check_name STRING, check_type STRING,  expected_value STRING, actual_value STRING, passed BOOLEAN,  severity STRING, details STRING) COMMENT 'Quality check results for ADOIT domain data products';CREATE OR REPLACE TABLE adoit_product.governance.product_observability (  product_name STRING, check_timestamp TIMESTAMP,  quality_score DOUBLE, freshness_hours DOUBLE,  sla_met BOOLEAN, row_count LONG, column_count INT,  schema_drift BOOLEAN, status STRING) COMMENT 'Observability metrics for ADOIT domain data products';

## Step 6: Tags for Discoverability

In [ ]:
%sqlALTER CATALOG adoit_product SET TAGS ('domain' = 'Enterprise Architecture', 'owner' = 'EA Team', 'data_mesh_role' = 'Domain Catalog');ALTER SCHEMA adoit_product.bronze     SET TAGS ('domain' = 'EA', 'layer' = 'bronze', 'owner' = 'EA Team');ALTER SCHEMA adoit_product.silver     SET TAGS ('domain' = 'EA', 'layer' = 'silver', 'owner' = 'EA Team');ALTER SCHEMA adoit_product.gold       SET TAGS ('domain' = 'EA', 'layer' = 'gold', 'owner' = 'EA Team');ALTER SCHEMA adoit_product.governance SET TAGS ('domain' = 'EA', 'layer' = 'governance', 'owner' = 'EA Team');ALTER TABLE adoit_product.gold.application_landscape SET TAGS (  'data_product' = 'true', 'domain' = 'EA', 'pii' = 'false',  'classification' = 'internal', 'sla_hours' = '4', 'quality_threshold' = '95');

## Step 7: Access Control

In [ ]:
%sqlGRANT USE CATALOG ON CATALOG adoit_product TO `account users`;GRANT USE SCHEMA ON CATALOG adoit_product TO `account users`;GRANT SELECT ON CATALOG adoit_product TO `account users`;

## ✅ Setup Complete- **Catalog**: `adoit_product` with EA domain metadata- **Schemas**: bronze, silver, gold, governance (4 schemas)- **Tables**: 3 bronze + 2 silver + 1 gold + 4 governance = **10 tables**- **Tags**: Catalog, schema, and table level- **Grants**: Account users can readNext: Run `01_Adoit_Pipeline` to populate the data product.